# Fuselage Design — As-Selected Re-Solve (COTS Hardware)
### Electric V-BAT-Like Tail-Sitter (EDF) — Conceptual Design Study

---

## Purpose

NB6 (`fuselage_design`) packaged the vehicle from mass-closure fractions
and *effective density estimates*. NB11 froze real hardware with catalog
masses and envelopes. This notebook **re-solves the fuselage with the
as-selected hardware** — downstream, as a new notebook, keeping the
pipeline one-way (ADR-0012). The conceptual `out/fuselage.yaml` remains
the geometry the CAD/CFD chain (NB8+) is built from; the deltas found here
are **findings** to fold back into `config/` in a deliberate next
iteration, never an automatic feedback edge.

Three things become real in this re-solve:

1. **Masses** — the COTS battery pack, FC + supporting avionics stack,
   motor + prop and ESC replace their fraction-derived allocations (the
   duct stays modeled: it is printed structure booked under propulsion).
2. **Packing** — the battery bay density comes from the frozen pack's own
   envelope, and each bay's length is **floored at the best-orientation
   axial length of the rigid part it must hold** (a volume-based bay
   assumes contents can be shaped to it; a rigid LiPo brick cannot).
3. **Fit** — every frozen part is checked against the space the re-solved
   hull actually gives it (reported, never filtered on).

## Axis Convention

Same as NB6: body FRD, stations from the nose tip +aft, $x_{body} = -x_s$.

## Inputs

- `out/components.yaml` *(NB11)* — frozen parts, envelopes, supporting stack
- `out/aileron_cots.yaml`, `out/vibration_cots.yaml` *(NB12/NB13)*
- `out/airfoil.yaml`, `out/control_vanes.yaml` *(NB2/NB3)*
- `out/fuselage.yaml` *(NB6 — conceptual solution, for the delta report)*
- `config/fuselage.yaml`, `config/modularity.yaml`

## Outputs

- `out/fuselage_cots.yaml` *(same schema as `out/fuselage.yaml` +
  `as_selected` / `bay_fit` / `delta_vs_conceptual` blocks)*
- `out/fuselage_layout_cots.png`

---

In [1]:
import sys, math
from pathlib import Path
from dataclasses import replace
import numpy as np
import matplotlib.pyplot as plt
import yaml

REPO_ROOT   = Path().resolve().parents[0]
SRC_PATH    = REPO_ROOT / "src"
CONFIG_PATH = REPO_ROOT / "config"
OUT_PATH    = REPO_ROOT / "out"
sys.path.insert(0, str(SRC_PATH))

plt.rcParams.update({
    "figure.dpi"        : 120,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"         : True,
    "grid.alpha"        : 0.25,
    "font.size"         : 10,
})
C = ["#2c7bb6", "#d7191c", "#1a9641", "#f68b33", "#762a83"]

FIG_DIR = Path("figures")   # per-notebook figures directory
FIG_DIR.mkdir(exist_ok=True)


# Section 1 — Design Inputs

Re-run the sizing loop from `config/` (same pattern as NB2–NB13) and load
the upstream handoffs: the as-selected aileron/vibration re-solves, the
frozen hardware list, and the conceptual fuselage for the delta report.

---

In [2]:
from conceptual_design import (
    run_sizing_loop, FrozenComponents,
    Environment, Mission, Aerodynamics, Battery,
    WeightFraction, PropulsiveSystemParameters,
)
from conceptual_design.forward_flight_power import ForwardFlightParams
from conceptual_design.wing_sizing import WingStructureParams
from conceptual_design.models import RotorParams, Avionics

env     = Environment()
mission = Mission.from_yaml(CONFIG_PATH / "mission.yaml")
aero    = Aerodynamics.from_yaml(CONFIG_PATH / "aerodynamics.yaml")
batt    = Battery.from_yaml(CONFIG_PATH / "battery.yaml")
wf      = WeightFraction.from_yaml(CONFIG_PATH / "initial_weight_fraction_estimation.yaml")
prop    = PropulsiveSystemParameters.from_yaml(CONFIG_PATH / "propulsive_system_parameters.yaml")
ff      = ForwardFlightParams.from_yaml(CONFIG_PATH / "forward_flight_params.yaml")
ws      = WingStructureParams.from_yaml(CONFIG_PATH / "wing_structure_params.yaml")
rotor    = RotorParams.from_yaml(CONFIG_PATH / "rotor.yaml")
avionics = Avionics.from_yaml(CONFIG_PATH / "avionics.yaml")

result = run_sizing_loop(
    m_payload_kg=mission.payload_kg, mission=mission, aero=aero, batt=batt,
    wf=wf, prop_params=prop, ff_params=ff, ws_params=ws, env=env,
    D_rotor_m=rotor.D_rotor_m, P_hotel_W=avionics.P_hotel_W,
)

af       = yaml.safe_load((OUT_PATH / "airfoil.yaml").read_text(encoding="utf-8"))
vanes    = yaml.safe_load((OUT_PATH / "control_vanes.yaml").read_text(encoding="utf-8"))
ail_cots = yaml.safe_load((OUT_PATH / "aileron_cots.yaml").read_text(encoding="utf-8"))
vib_cots = yaml.safe_load((OUT_PATH / "vibration_cots.yaml").read_text(encoding="utf-8"))
fus_est  = yaml.safe_load((OUT_PATH / "fuselage.yaml").read_text(encoding="utf-8"))
comps    = FrozenComponents.from_yaml(OUT_PATH / "components.yaml")

MTOW_kg      = result.m_total_kg
chord_mean_m = result.wing.chord_mean
S_wing       = result.wing.S_wing
R_hub_m      = vanes["R_hub_m"]
m_aileron_servo_kg   = ail_cots["servo_mass_kg_each"]   * ail_cots["n_ailerons"]
m_aileron_linkage_kg = ail_cots["linkage_mass_kg_each"] * ail_cots["n_ailerons"]
m_isolation_avionics_kg = vib_cots["m_isolation_avionics_kg"]
m_isolation_struct_kg   = vib_cots["m_isolation_struct_kg"]
sway_pad_m              = vib_cots["sway_pad_total_m"]

print(f"Converged MTOW (closure) : {MTOW_kg:.3f} kg")
print(f"Frozen hardware          : " + ", ".join(
    f"{comps[c].id}" for c in ("battery", "flight_controller", "esc", "edf_motor", "propeller", "servo")))
print(f"Aileron hw (NB12)        : {(m_aileron_servo_kg + m_aileron_linkage_kg)*1e3:.1f} g")
print(f"Isolation hw (NB13)      : {(m_isolation_avionics_kg + m_isolation_struct_kg)*1e3:.1f} g, "
      f"sway pad {sway_pad_m*1e3:.2f} mm")


Converged MTOW (closure) : 2.303 kg
Frozen hardware          : gens_ace_6s_4000_30c, pixhawk_6c, apd_80f3x, sunnysky_x4120_465, ma_3blade_8x6, kst_x08_v6
Aileron hw (NB12)        : 23.8 g
Isolation hw (NB13)      : 32.0 g, sway pad 1.60 mm


# Section 2 — As-Selected Masses and Envelopes

Replace the fraction-derived estimates with the frozen hardware
(`cots_integration`):

- **battery** — catalog mass, and packing density from its own envelope
  (mass / box volume) instead of the configured `rho_battery_pack`;
- **avionics** — the "fraction" is rebuilt bottom-up (FC + supporting
  stack + the servo/isolator carve-outs), so `size_fuselage`'s ADR-0005
  carve arithmetic nets back to the *actual* bay content;
- **propulsion** — actual motor + prop and ESC masses at their layout
  stations (duct stays modeled);
- **bay floors** — the battery pack and FC board envelopes must fit their
  bays in some orientation.

---

In [3]:
from conceptual_design.fuselage_design import (
    FuselageParams, ModularityParams, size_fuselage, fuselage_radius,
)
from conceptual_design.cots_integration import (
    avionics_budget_bottom_up, bay_fit_report, bay_part_envelopes,
    effective_density, propulsion_item_masses,
)

fus_p = FuselageParams.from_yaml(CONFIG_PATH / "fuselage.yaml")
mod_p = ModularityParams.from_yaml(CONFIG_PATH / "modularity.yaml")

battery = comps["battery"]
fc      = comps["flight_controller"]
servo   = comps["servo"]

rho_batt_eff = effective_density(battery)
m_avionics_cots = avionics_budget_bottom_up(
    m_fc_kg=fc.mass_kg, m_supporting_kg=comps.supporting_mass_kg,
    n_vane_servos=int(vanes["n_vanes"]), n_aileron_servos=int(ail_cots["n_ailerons"]),
    m_servo_each_kg=servo.mass_kg,
    m_isolation_avionics_kg=m_isolation_avionics_kg,
)
prop_masses = propulsion_item_masses(result.m_propulsion_kg, comps)
envelopes   = bay_part_envelopes(comps)

fus_p_cots = replace(fus_p,
                     rho_battery_pack=rho_batt_eff,
                     servo_mass_kg=servo.mass_kg)

print(f"{'quantity':<34}{'closure/est.':>14}{'as-selected':>14}")
print("-" * 62)
print(f"{'battery mass [g]':<34}{result.m_battery_kg*1e3:>14.1f}{battery.mass_kg*1e3:>14.1f}")
print(f"{'battery pack density [kg/m^3]':<34}{fus_p.rho_battery_pack:>14.0f}{rho_batt_eff:>14.0f}")
print(f"{'avionics budget [g]':<34}{result.m_avionics_kg*1e3:>14.1f}{m_avionics_cots*1e3:>14.1f}")
print(f"{'motor + prop [g]':<34}{0.60*result.m_propulsion_kg*1e3:>14.1f}{prop_masses['motor_fan']*1e3:>14.1f}")
print(f"{'ESC [g]':<34}{0.25*result.m_propulsion_kg*1e3:>14.1f}{prop_masses['esc']*1e3:>14.1f}")
print(f"{'vane/aileron servo, each [g]':<34}{fus_p.servo_mass_kg*1e3:>14.1f}{servo.mass_kg*1e3:>14.1f}")
print()
for bay, envs in envelopes.items():
    for shape, dims in envs:
        print(f"bay floor: {bay:<9} <- {shape} " +
              " x ".join(f"{d*1e3:.1f}" for d in dims) + " mm")


quantity                            closure/est.   as-selected
--------------------------------------------------------------
battery mass [g]                           554.6         613.0
battery pack density [kg/m^3]               1500          2601
avionics budget [g]                        184.2         208.0
motor + prop [g]                           207.2         306.0
ESC [g]                                     86.3          30.0
vane/aileron servo, each [g]                12.0           8.9

bay floor: battery   <- box 137.0 x 43.0 x 40.0 mm
bay floor: avionics  <- box 84.8 x 44.0 x 12.4 mm


# Section 3 — Re-Solve the Hull

Same physics as NB6 (`size_fuselage`, one call) with the as-selected
masses, the envelope floors in the diameter/stack solve, and the actual
propulsion item masses in the layout. The wing is again placed for the
configured static margin **by construction** — what moves is everything
around it.

The as-selected all-up mass is the layout total: the closure MTOW plus
every standing mass finding of the freeze (battery quantisation, motor
class, avionics stack) in one number.

---

In [4]:
fus = size_fuselage(
    m_battery_kg    = battery.mass_kg,
    m_payload_kg    = result.m_payload_kg,
    m_avionics_kg   = m_avionics_cots,
    m_propulsion_kg = result.m_propulsion_kg,
    m_structure_kg  = result.m_structure_kg,
    m_wing_kg       = result.m_wing_kg,
    m_misc_kg       = result.m_misc_kg,
    R_hub_m         = R_hub_m,
    D_rotor_m       = rotor.D_rotor_m,
    c_vane_m        = vanes["c_vane_m"],
    n_vanes         = vanes["n_vanes"],
    S_vane_m2       = vanes["S_vane_m2"],
    hinge_xc        = vanes["hinge_xc"],
    chord_mean_m    = chord_mean_m,
    m_aileron_servo_kg   = m_aileron_servo_kg,
    m_aileron_linkage_kg = m_aileron_linkage_kg,
    m_isolation_avionics_kg = m_isolation_avionics_kg,
    m_isolation_struct_kg   = m_isolation_struct_kg,
    sway_pad_m              = sway_pad_m,
    V_cruise        = mission.V_cruise,
    rho             = env.rho,
    S_wing          = S_wing,
    p               = fus_p_cots,
    mod             = mod_p,
    construction_method = wf.construction_method,
    part_envelopes   = envelopes,
    prop_item_masses = prop_masses,
)

m_asselected = sum(it.mass_kg for it in fus.items)
m_delta      = m_asselected - MTOW_kg

print(f"Active constraint : {fus.active_constraint.upper()}")
print(f"D_fus             : {fus.D_fus*1e3:7.1f} mm   (conceptual {fus_est['D_fus_m']*1e3:.1f} mm)")
print(f"L_fus             : {fus.L_fus*1e3:7.1f} mm   (conceptual {fus_est['L_fus_m']*1e3:.1f} mm)")
print(f"  nose / mid / tail : {fus.L_nose*1e3:.0f} / {fus.L_mid*1e3:.0f} / {fus.L_tail*1e3:.0f} mm")
print(f"Bay stack         : {fus.L_stack_m*1e3:7.1f} mm  of  {fus.L_avail_m*1e3:.1f} mm available "
      f"({fus.L_stack_m/fus.L_avail_m*100:.0f}% used)")
print(f"CG station        : {fus.x_CG*1e3:7.1f} mm   (conceptual {fus_est['x_CG_m']*1e3:.1f} mm)")
print(f"Structure ({fus.construction_method}) : {fus.m_shell_kg*1e3:.1f} g "
      f"of {fus.m_struct_budget_kg*1e3:.1f} g budget")
print()
print(f"As-selected all-up mass : {m_asselected:.3f} kg  "
      f"(closure MTOW {MTOW_kg:.3f} kg, {m_delta*1e3:+.0f} g of standing findings)")


Active constraint : PACKAGING
D_fus             :   106.3 mm   (conceptual 95.7 mm)
L_fus             :   531.3 mm   (conceptual 478.3 mm)
  nose / mid / tail : 117 / 255 / 159 mm
Bay stack         :   313.5 mm  of  313.5 mm available (100% used)
CG station        :   256.8 mm   (conceptual 236.6 mm)
Structure (segmented_fdm) : 374.9 g of 341.5 g budget

As-selected all-up mass : 2.427 kg  (closure MTOW 2.303 kg, +125 g of standing findings)


# Section 4 — Physical Fit of the Frozen Hardware

Each frozen part against the space the re-solved hull gives it:
best-orientation axial length (+ clearance) vs. its bay, the ESC vs. its
mid-body slot, the motor's diameter vs. the tail centerbody. **Findings,
never filters** — same discipline as the NB11 mass-allocation report.

---

In [5]:
fit = bay_fit_report(fus, fus_p_cots, comps)

print(f"{'part':<22}{'where':<28}{'need [mm]':>10}{'have [mm]':>10}{'fit':>6}")
print("-" * 76)
for e in fit:
    need = f"{e['need_mm']:.1f}" if e["need_mm"] is not None else "n/a"
    print(f"{e['part']:<22}{e['where']:<28}{need:>10}{e['have_mm']:>10.1f}"
          f"{'  ok' if e['ok'] else '  NO':>6}")

n_misfit = sum(1 for e in fit if not e["ok"])
print()
print("All frozen parts fit the re-solved hull." if n_misfit == 0 else
      f"{n_misfit} part(s) do NOT fit -- standing finding(s) against the packaging assumptions.")


part                  where                        need [mm] have [mm]   fit
----------------------------------------------------------------------------
gens_ace_6s_4000_30c  battery                          142.0     142.0    ok
pixhawk_6c            avionics                          17.4      49.6    ok
apd_80f3x             esc                               17.0      40.0    ok
sunnysky_x4120_465    tail centerbody (2 x r_hub)       46.0      81.2    ok

All frozen parts fit the re-solved hull.


# Section 5 — As-Selected Internal Layout

Same side-view as NB6, drawn from the re-solved geometry. The battery bay
is now as long as the real pack; the CG marker is the as-selected one.

---

In [6]:
xs = np.linspace(0.0, fus.L_fus, 400)
rr = np.array([fuselage_radius(x, fus.D_fus, fus.L_fus,
                               fus_p_cots.f_nose, fus_p_cots.f_tail, fus.r_hub) for x in xs])
by_name = {it.name: it for it in fus.items}

fig, ax = plt.subplots(figsize=(11, 5))
mm = 1e3

# clamshell joint line (ADR-0010)
ax.plot([0, fus.x_clam_aft_m*mm], [-fus.D_fus/2*mm - 6]*2,
        color=C[3], lw=3, solid_capstyle='butt', zorder=6)
ax.axvline(fus.x_clam_aft_m*mm, color=C[3], ls='-.', lw=1.4, zorder=6)
ax.annotate('clamshell lid (hinged)', (fus.x_clam_aft_m*mm/2, -fus.D_fus/2*mm - 14),
            ha='center', fontsize=8, color=C[3])

# fuselage outline
ax.fill_between(xs*mm, -rr*mm, rr*mm, color="#d9e4ee", zorder=1)
ax.plot(xs*mm,  rr*mm, color=C[0], lw=1.8, zorder=3)
ax.plot(xs*mm, -rr*mm, color=C[0], lw=1.8, zorder=3)

# internal bays
bay_colors = {"payload": C[2], "avionics": C[4], "battery": C[3], "esc": C[1]}
r_int = fus.D_fus/2 - fus_p_cots.t_shell_m
for it in fus.items:
    if it.name in bay_colors and it.length > 0:
        ax.add_patch(plt.Rectangle((it.x_start*mm, -0.78*r_int*mm),
                                   it.length*mm, 1.56*r_int*mm,
                                   facecolor=bay_colors[it.name], alpha=0.55,
                                   edgecolor="k", lw=0.5, zorder=4))
        ax.annotate(it.name, ((it.x_start + it.length/2)*mm, 0),
                    ha="center", va="center", fontsize=8, rotation=90, zorder=6)

# duct annulus (section view)
x_d0 = (by_name["duct"].x_cg - fus.duct_chord/2) * mm
for sgn in (+1, -1):
    ax.add_patch(plt.Rectangle((x_d0, sgn*fus.D_duct_inner/2*mm),
                               fus.duct_chord*mm, sgn*fus_p_cots.t_duct_m*mm,
                               facecolor="#666", edgecolor="k", lw=0.5, zorder=4))
ax.annotate("duct", (x_d0 + fus.duct_chord*mm/2, fus.D_duct_outer/2*mm + 6),
            ha="center", fontsize=8)

# control vanes (T/B pair in side view)
for sgn in (+1, -1):
    ax.add_patch(plt.Rectangle(((fus.x_vane - vanes["c_vane_m"]/2)*mm,
                                sgn*vanes["R_hub_m"]*mm),
                               vanes["c_vane_m"]*mm,
                               sgn*(vanes["R_tip_m"]-vanes["R_hub_m"])*mm,
                               facecolor=C[1], alpha=0.8, edgecolor="k",
                               lw=0.5, zorder=4))
ax.annotate("vanes", (fus.x_vane*mm, vanes["R_tip_m"]*mm + 6),
            ha="center", fontsize=8, color=C[1])

# wing chord (side view)
ax.plot([fus.x_wing_LE*mm, (fus.x_wing_LE + chord_mean_m)*mm], [0, 0],
        color=C[2], lw=5, solid_capstyle="butt", zorder=5)
ax.annotate("wing chord", ((fus.x_wing_LE + chord_mean_m/2)*mm, -14),
            ha="center", fontsize=8, color=C[2])

# fan plane
x_fan = by_name["motor_fan"].x_cg * mm
ax.plot([x_fan, x_fan], [-fus.r_hub*mm, fus.r_hub*mm], color="k", ls="-.", lw=1)
ax.annotate("fan", (x_fan, -fus.r_hub*mm - 10), ha="center", fontsize=8)

# CG
ax.plot(fus.x_CG*mm, 0, "o", color="k", ms=9, zorder=7,
        markerfacecolor="w", markeredgewidth=2)
ax.axvline(fus.x_CG*mm, color="k", ls="--", lw=0.8, alpha=0.6)
ax.annotate(f"CG  (x_s = {fus.x_CG*mm:.0f} mm)", (fus.x_CG*mm, fus.D_fus/2*mm + 18),
            ha="center", fontsize=9, fontweight="bold")

ax.set_xlabel(r"station from nose, +aft  [mm]     ($x_{body} = -x_s$,  FRD)")
ax.set_ylabel("z  [mm]")
ax.set_title("As-selected fuselage layout — side view (COTS re-solve)")
ax.set_aspect("equal")
ax.set_xlim(-30, (fus.x_vane + 0.06)*mm)
fig.tight_layout()
fig.savefig(OUT_PATH / "fuselage_layout_cots.png", bbox_inches="tight")
plt.show()

print(f"{'item':<14}{'mass [kg]':>10}{'x_cg [mm]':>12}")
for it in fus.items:
    print(f"{it.name:<14}{it.mass_kg:>10.3f}{it.x_cg*1e3:>12.1f}")
print("-" * 36)
print(f"{'total':<14}{m_asselected:>10.3f}{fus.x_CG*1e3:>12.1f}")


item           mass [kg]   x_cg [mm]
payload            0.500        90.2
avionics           0.139       175.9
battery            0.613       271.7
esc                0.030       265.7
motor_fan          0.306       491.5
duct               0.052       505.2
shell_struct       0.342       239.1
battery_tray       0.015       271.7
control_hw         0.061       551.5
isolation_hw       0.032       133.0
joint_hw           0.020       186.0
misc               0.146       212.5
wing               0.114       256.8
aileron_hw         0.024       256.8
spar_hw            0.034       256.8
------------------------------------
total              2.427       256.8


D:\Temp\ipykernel_28352\1589013496.py:78: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Section 6 — Drag Budget and Vane Arm Re-Check

The hull that holds the real hardware is not the hull NB6 drew — its
wetted area and $C_{D0}$ move, and so does the CG-to-vane arm the NB3
sizing rests on. Re-check both against the same budgets as NB6.

---

In [7]:
CD0_budget   = aero.CD0
CD0_wing_est = af["Cd0_section"] * 1.10
CD0_misc     = 0.0025    # duct external + vanes + antennas + interference (as NB6)
CD0_total    = CD0_wing_est + fus.CD0_fus + CD0_misc
drag_margin  = (CD0_budget - CD0_total) / CD0_budget * 100

print(f"{'CD0 fuselage (as-selected)':<34}: {fus.CD0_fus:.5f}   (conceptual {fus_est['CD0_fus']:.5f})")
print(f"{'CD0 total (built up)':<34}: {CD0_total:.5f}")
print(f"{'CD0 assumed in sizing (NB1)':<34}: {CD0_budget:.5f}")
print(f"Drag budget {'CLOSES' if CD0_total <= CD0_budget else 'EXCEEDED -- revisit aerodynamics.yaml'}"
      f"  (margin {drag_margin:+.1f}%)")
print()
L_assumed = vanes["L_CG_m"]
factor    = fus.L_vane_arm / L_assumed
print(f"{'Vane-to-CG arm (as-selected)':<34}: {fus.L_vane_arm*1e3:.1f} mm "
      f"(conceptual {fus_est['L_vane_arm_m']*1e3:.1f} mm)")
print(f"{'Control authority vs NB3 assumption':<34}: x{factor:.2f}")
assert factor >= 1.0, (
    "as-selected vane arm shorter than the NB3 sizing assumption -- "
    "re-visit L_CG_m in config")


CD0 fuselage (as-selected)        : 0.00629   (conceptual 0.00533)
CD0 total (built up)              : 0.02265
CD0 assumed in sizing (NB1)       : 0.02500
Drag budget CLOSES  (margin +9.4%)

Vane-to-CG arm (as-selected)      : 299.9 mm (conceptual 267.0 mm)
Control authority vs NB3 assumption: x2.60


# Section 7 — Output Export

`out/fuselage_cots.yaml` — same schema as `out/fuselage.yaml` plus the
`as_selected` mass rollup, the `bay_fit` report, and the
`delta_vs_conceptual` block (pinned by the regression tests). Consumed by
the design summary (NB15); **not** by the CAD/CFD chain, which stays on
the conceptual geometry until a config revision adopts these findings.

---

In [8]:
from conceptual_design.fuselage_design import write_fuselage_yaml

delta_vs_conceptual = {
    "D_fus_mm":      round((fus.D_fus - fus_est["D_fus_m"]) * 1e3, 2),
    "L_fus_mm":      round((fus.L_fus - fus_est["L_fus_m"]) * 1e3, 2),
    "x_CG_mm":       round((fus.x_CG - fus_est["x_CG_m"]) * 1e3, 2),
    "S_wet_m2":      round(fus.S_wet_m2 - fus_est["S_wet_m2"], 5),
    "CD0_fus":       round(fus.CD0_fus - fus_est["CD0_fus"], 5),
    "L_vane_arm_mm": round((fus.L_vane_arm - fus_est["L_vane_arm_m"]) * 1e3, 2),
}

write_fuselage_yaml(
    fus, fus_p_cots, OUT_PATH / "fuselage_cots.yaml",
    regen_notebook="notebooks/fuselage_design_cots.ipynb",
    extra={
        "as_selected": {
            "m_items_total_kg":     round(m_asselected, 5),
            "m_closure_mtow_kg":    round(MTOW_kg, 5),
            "m_delta_kg":           round(m_delta, 5),
            "rho_battery_eff_kgm3": round(rho_batt_eff, 1),
            "battery_id": battery.id, "fc_id": fc.id, "servo_id": servo.id,
            "edf_motor_id": comps["edf_motor"].id,
            "esc_id": comps["esc"].id, "propeller_id": comps["propeller"].id,
        },
        "bay_fit": fit,
        "delta_vs_conceptual": delta_vs_conceptual,
    },
)
print(f"As-selected fuselage design written -> {OUT_PATH / 'fuselage_cots.yaml'}")


As-selected fuselage design written -> D:\Dev\vbat-uav-notebooks\out\fuselage_cots.yaml


# Section 8 — Design Summary

---

In [9]:
bar = "=" * 62
print(bar)
print("  FUSELAGE AS-SELECTED RE-SOLVE SUMMARY".center(62))
print(bar)
print(f"  {'quantity':<30}{'NB6 (est.)':>13}{'this NB':>13}")
print("  " + "-" * 58)
print(f"  {'D_fus [mm]':<30}{fus_est['D_fus_m']*1e3:>13.1f}{fus.D_fus*1e3:>13.1f}")
print(f"  {'L_fus [mm]':<30}{fus_est['L_fus_m']*1e3:>13.1f}{fus.L_fus*1e3:>13.1f}")
print(f"  {'CG station [mm]':<30}{fus_est['x_CG_m']*1e3:>13.1f}{fus.x_CG*1e3:>13.1f}")
print(f"  {'S_wet [m^2]':<30}{fus_est['S_wet_m2']:>13.4f}{fus.S_wet_m2:>13.4f}")
print(f"  {'CD0 fuselage':<30}{fus_est['CD0_fus']:>13.5f}{fus.CD0_fus:>13.5f}")
print(f"  {'vane arm [mm]':<30}{fus_est['L_vane_arm_m']*1e3:>13.1f}{fus.L_vane_arm*1e3:>13.1f}")
print()
print(f"  {'As-selected all-up mass':<30}: {m_asselected:.3f} kg  ({m_delta*1e3:+.0f} g vs closure)")
print(f"  {'Bay fit':<30}: " +
      ("all frozen parts fit" if all(e['ok'] for e in fit)
       else f"{sum(1 for e in fit if not e['ok'])} misfit(s) -- see Section 4"))
print(f"  {'Static margin':<30}: {fus_p_cots.static_margin*100:.1f} % MAC (by construction)")
print(bar)
print("  The conceptual geometry (out/fuselage.yaml) remains the CAD/CFD")
print("  baseline; adopting these deltas is a config revision (ADR-0012).")
print(bar)


             FUSELAGE AS-SELECTED RE-SOLVE SUMMARY            
  quantity                         NB6 (est.)      this NB
  ----------------------------------------------------------
  D_fus [mm]                             95.7        106.3
  L_fus [mm]                            478.3        531.3
  CG station [mm]                       236.6        256.8
  S_wet [m^2]                          0.1356       0.1652
  CD0 fuselage                        0.00533      0.00629
  vane arm [mm]                         267.0        299.9

  As-selected all-up mass       : 2.427 kg  (+125 g vs closure)
  Bay fit                       : all frozen parts fit
  Static margin                 : 5.0 % MAC (by construction)
  The conceptual geometry (out/fuselage.yaml) remains the CAD/CFD
  baseline; adopting these deltas is a config revision (ADR-0012).
